# Jauge-02 — De la symétrie globale à la jauge locale : U(1) puis SU(2)

**Jauge-01** a établi le pont symétrie globale → quantité conservée (théorème de Noether). Ce carnet franchit l'étape suivante de l'arc : que se passe-t-il quand on exige la symétrie **en chaque point de l'espace-temps séparément** ?

C'est l'idée de **jauge locale**, introduite pour le spin isotopique par Yang et Mills (1954) par analogie explicite avec la phase du champ chargé (copie archivée au gisement, cf. Références) :

1. **U(1)** — un champ scalaire complexe est invariant sous une phase *constante* ; rendre la phase *locale* $\alpha(x,t)$ casse l'invariance, et la restaurer force l'introduction d'un champ de jauge $A_\mu$ via la dérivée covariante $D_\mu$ ;
2. **SU(2)** — la version non abélienne de Yang–Mills : les champs de jauge eux-mêmes portent la charge, le tenseur de courbure gagne un terme d'**auto-interaction** $\varepsilon^{abc} W^b W^c$, vérifié ici par le calcul exact du commutateur $[D_\mu, D_\nu]$ ;
3. **Contrôles** — invariance numérique de $\mathrm{Tr}(F_{\mu\nu} F^{\mu\nu})$ sous une transformation de jauge SU(2) explicite, et contre-témoin abélien qui montre que le pipeline détecte une transformation incomplète.

**Filière** : SymmetryGroups/Jauge — prolonge [Jauge-01](Jauge-01-Theoreme-Noether-Python.ipynb).

## 1. Symétrie U(1) globale d'un champ scalaire complexe

Champ complexe $\varphi = u + iv$ en dimension $1+1$ ($t, x$), Lagrangien du champ libre :

$$\mathcal{L} = \partial_\mu \varphi^* \, \partial^\mu \varphi - m^2 \varphi^* \varphi .$$

La transformation $\varphi \to e^{i\alpha}\varphi$ avec $\alpha$ **constant** est la symétrie de jauge globale. On la vérifie exactement en substituant $u \to u\cos\alpha - v\sin\alpha$, $v \to u\sin\alpha + v\cos\alpha$.

In [1]:
import sympy as sp

t, x, m, q, g = sp.symbols('t x m q g', positive=True, real=True)
u = sp.Function('u', real=True)(t, x)
v = sp.Function('v', real=True)(t, x)
A_t = sp.Function('A_t', real=True)(t, x)
A_x = sp.Function('A_x', real=True)(t, x)
eta = sp.diag(1, -1)  # metrique (t, x)

# Lagrangien du champ libre (phi = u + I v, sans champ de jauge)
def d_free(f, mu):
    # derivee partielle selon la coordonnee d'indice mu (0 = t, 1 = x)
    return sp.diff(f, t) if mu == 0 else sp.diff(f, x)

phi = u + sp.I * v
L_free = sum(eta[mu, mu] * d_free(sp.conjugate(phi), mu) * d_free(phi, mu)
             for mu in range(2)) - m**2 * sp.conjugate(phi) * phi
L_free = sp.expand(L_free)

# Transformation de phase globale : alpha constant
alpha = sp.symbols('alpha', real=True)
u_g = u * sp.cos(alpha) - v * sp.sin(alpha)
v_g = u * sp.sin(alpha) + v * sp.cos(alpha)
L_global = L_free.subs({u: u_g, v: v_g}, simultaneous=True).doit()
delta_global = sp.simplify(sp.expand(L_global - L_free))
print("delta L (phase globale, alpha constant) =", delta_global)

delta L (phase globale, alpha constant) = 0


### Lecture du résultat

La phase globale laisse le lagrangien **inchangé** : `delta L (phase globale, alpha constant) = 0`, exactement et non approximativement. La raison tient en une ligne : sous $\phi \to e^{i\alpha}\phi$ avec $\alpha$ **constant**, le terme de masse $m^2\phi^*\phi$ est inchangé parce que $e^{i\alpha}e^{-i\alpha} = 1$, et le terme cinétique $\partial_\mu\phi^*\partial^\mu\phi$ l'est aussi parce que la dérivée d'une exponentielle constante est nulle. C'est la symétrie $U(1)$ **globale** du champ scalaire complexe : elle ne fait intervenir aucun champ auxiliaire, le lagrangien ne dépend pas du tout de $\alpha$, et le courant conservé associé — la charge $U(1)$ — est celui que le carnet Jauge-01 a établi.

## 2. Le problème : rendre la phase locale

Rendons maintenant $\alpha$ une **fonction** $\alpha(x,t)$ : chaque point de l'espace-temps choisit sa propre phase. C'est l'exigence de Yang–Mills transposée au champ chargé (« the arbitrariness in choosing the phase factor is local in character », copie archivée p. 192).

In [2]:
# Phase locale : alpha devient une fonction alpha(t, x)
a_loc = sp.Function('alpha', real=True)(t, x)
u_l = u * sp.cos(a_loc) - v * sp.sin(a_loc)
v_l = u * sp.sin(a_loc) + v * sp.cos(a_loc)
L_local = L_free.subs({u: u_l, v: v_l}, simultaneous=True).doit()

delta_local = sp.trigsimp(sp.expand(L_local - L_free))
# On isole les termes de derivee de alpha (ils ne se compensent pas)
print("delta L (phase locale) -- expression complete, facteurs en d(alpha) :")
print("  ", sp.collect(sp.trigsimp(delta_local), [sp.diff(a_loc, t), sp.diff(a_loc, x)]))
# Complement de la mesure : on neutralise les derivees de alpha, ce qui
# ramene la phase LOCALE a la phase GLOBALE -- et le Lagrangien doit alors
# etre invariant. Toute la rupture vient donc de d(alpha), rien d'autre.
delta_sans_derivees = sp.trigsimp(sp.expand(delta_local).subs(
    {sp.Derivative(a_loc, t): 0, sp.Derivative(a_loc, x): 0}))
print("delta L (d(alpha) neutralise -- phase globale) =", delta_sans_derivees)

delta L (phase locale) -- expression complete, facteurs en d(alpha) :
   (2*u(t, x)*Derivative(v(t, x), t) - 2*v(t, x)*Derivative(u(t, x), t))*Derivative(alpha(t, x), t) + (-2*u(t, x)*Derivative(v(t, x), x) + 2*v(t, x)*Derivative(u(t, x), x))*Derivative(alpha(t, x), x) + (-u(t, x)**2 - v(t, x)**2)*Derivative(alpha(t, x), x)**2 + (u(t, x)**2 + v(t, x)**2)*Derivative(alpha(t, x), t)**2
delta L (d(alpha) neutralise -- phase globale) = 0


### Lecture du résultat

La phase locale produit une expression **non nulle**, et sa structure le dit précisément : elle ne dépend que des **dérivées** de $\alpha$, jamais de $\alpha$ lui-même. Le coefficient de $(\partial_t\alpha)^2$ vaut $+|\phi|^2$ et celui de $(\partial_x\alpha)^2$ vaut $-|\phi|^2$ — c'est la signature $(+,-)$ qui décide du signe. Les termes croisés associent $\partial_t\alpha$ à $u\,\partial_t v - v\,\partial_t u$ et $\partial_x\alpha$ à $u\,\partial_x v - v\,\partial_x u$, avec le signe opposé : la partie réelle et la partie imaginaire ne se transforment pas indépendamment.

Le test décisif est la dernière ligne : une fois les dérivées de $\alpha$ neutralisées — ce qui ramène la phase locale à la phase globale — l'écart redevient **exactement 0**. Toute la rupture vient donc de $\partial_\mu\alpha$, et de rien d'autre. Les termes en $\partial_\mu\alpha$ **ne se compensent pas** : la symétrie globale ne survit pas à la localisation. C'est le point de départ historique de toute l'électrodynamique de jauge.

## 3. Dérivée covariante et champ de jauge

La réparation : remplacer $\partial_\mu$ par la **dérivée covariante**

$$D_\mu = \partial_\mu + iq A_\mu, \qquad A_\mu \to A_\mu - \frac{1}{q}\partial_\mu \alpha,$$

et reconstruire $\mathcal{L}_{\text{jauge}} = (D_\mu\varphi)^* (D^\mu\varphi) - m^2\varphi^*\varphi$. Vérifions d'abord **exactement** que $D_\mu\varphi$ se transforme comme $\varphi$ lui-même.

In [3]:
# Propriete de covariance de la derivee covariante (exacte)
phi = u + sp.I * v
Dphi_t = sp.diff(phi, t) + sp.I * q * A_t * phi
Dphi_x = sp.diff(phi, x) + sp.I * q * A_x * phi

# Cote transforme : phi' = exp(i*alpha)*phi, A' = A - d(alpha)/q
exp_i = sp.exp(sp.I * a_loc)
A_t_p = A_t - sp.diff(a_loc, t) / q
A_x_p = A_x - sp.diff(a_loc, x) / q

Dphi_t_p = sp.diff(exp_i * phi, t) + sp.I * q * A_t_p * exp_i * phi
Dphi_x_p = sp.diff(exp_i * phi, x) + sp.I * q * A_x_p * exp_i * phi

cov_t = sp.simplify(sp.expand(Dphi_t_p - exp_i * Dphi_t))
cov_x = sp.simplify(sp.expand(Dphi_x_p - exp_i * Dphi_x))
print("D'_t(phi') - e^(i alpha) D_t(phi) =", cov_t)
print("D'_x(phi') - e^(i alpha) D_x(phi) =", cov_x)

D'_t(phi') - e^(i alpha) D_t(phi) = 0
D'_x(phi') - e^(i alpha) D_x(phi) = 0


**De la covariance à l'invariance.** La covariance — $D_\mu\varphi$ se transforme comme $\varphi$ lui-même — est une condition **nécessaire**, pas suffisante : elle dit que le terme de couplage se comporte correctement, pas que le lagrangien est invariant. Passons à la vérification **complète** : le lagrangien couplé, masse comprise, doit être inchangé par la transformation de jauge.

In [4]:
# Invariance de jauge EXACTE du Lagrangien couple.
# On ecrit la transformation en forme COMPLEXE (exp(i alpha)) plutot qu'en
# composantes reelles (cos, sin) : l'algebre reste en exponentielles, la
# simplification reste rapide, et le resultat attendu est exactement nul.

def lagrange_jauge(At, Ax, ph):
    # D_mu ph = d_mu ph + i q A_mu ph ; L = |D_t ph|^2 - |D_x ph|^2 - m^2 |ph|^2
    Dt = sp.diff(ph, t) + sp.I * q * At * ph
    Dx = sp.diff(ph, x) + sp.I * q * Ax * ph
    return sp.expand(sp.conjugate(Dt) * Dt - sp.conjugate(Dx) * Dx
                     - m**2 * sp.conjugate(ph) * ph)

L_couplage = lagrange_jauge(A_t, A_x, phi)
L_couplage_p = lagrange_jauge(A_t - sp.diff(a_loc, t) / q,
                              A_x - sp.diff(a_loc, x) / q,
                              sp.exp(sp.I * a_loc) * phi)
delta_jauge = sp.simplify(sp.expand(L_couplage_p - L_couplage))
print("delta L_jauge (transformation locale complete) =", delta_jauge)

delta L_jauge (transformation locale complete) = 0


### Lecture du résultat

L'invariance de jauge est **exacte**, pas approchée : `delta L_jauge (transformation locale complete) = 0`. Deux effets se compensent exactement. D'un côté, la dérivée covariante gagne $\pm iqA_\mu\phi$ ; de l'autre, le potentiel de jauge se transforme en $A_\mu \to A_\mu - \tfrac1q\partial_\mu\alpha$, et la dérivée de $\alpha$ qui apparaît ainsi compense le terme précédent. Ni la masse, ni le terme cinétique ne laissent de résidu.

C'est là que se joue la différence avec la symétrie globale : pour la phase globale, le lagrangien ne dépendait **pas du tout** de $\alpha$ ; pour la phase locale, il ne dépend plus que de $\partial_\mu\alpha$ — et cette dépendance a été absorbée par le champ auxiliaire $A_\mu$, qui devient une partie prenante du modèle et non un artifice de calcul.

## 4. Le tenseur $F_{\mu\nu}$ et le Lagrangien complet

Le champ $A_\mu$ a lui-même une dynamique. Son tenseur de courbure abélien $F_{\mu\nu} = \partial_\mu A_\nu - \partial_\nu A_\mu$ est **invariant** sous la jauge ; le terme cinétique $-\tfrac14 F_{\mu\nu}F^{\mu\nu}$ complète le Lagrangien scalaire de l'électrodynamique (sqED en 1+1 dimensions).

In [5]:
# F est invariant sous la transformation de jauge abelienne
F_tx = sp.diff(A_x, t) - sp.diff(A_t, x)
F_tx_p = F_tx.subs({A_t: A_t_p, A_x: A_x_p}, simultaneous=True).doit()
print("F' - F =", sp.simplify(sp.expand(F_tx_p - F_tx)))

# Lagrangien complet : champ couple + terme cinetique du champ de jauge.
# En 1+1 D, -1/4 F_mu nu F^mu nu = +F_tx^2 / 2 (metrique +,-)
L_F = sp.Rational(1, 2) * (sp.diff(A_x, t) - sp.diff(A_t, x))**2
L_total = sp.expand(L_couplage + L_F)
L_total_p = sp.expand(
    lagrange_jauge(A_t_p, A_x_p, sp.exp(sp.I * a_loc) * phi)
    + sp.Rational(1, 2) * (sp.diff(A_x_p, t) - sp.diff(A_t_p, x))**2)
print("delta L_total (jauge locale) =",
      sp.simplify(sp.expand(L_total_p - L_total)))

F' - F = 0
delta L_total (jauge locale) = 0


### Lecture du résultat

Le champ $F_{\mu\nu}$ est **invariant** sous la jauge : `F' - F = 0`, exactement. La raison tient en une ligne — dans $\partial_\mu A'_\nu - \partial_\nu A'_\mu$, les termes en $\partial_\mu\partial_\nu\alpha$ s'annulent (dérivées secondes symétriques) et il ne reste que des termes en $\partial_\mu\alpha$ qui se compensent entre eux.

Et surtout, le **Lagrangien complet** est lui aussi invariant : `delta L_total (jauge locale) = 0`, une fois le terme cinétique $-\tfrac14 F_{\mu\nu}F^{\mu\nu}$ ajouté au lagrangien couplé. C'est le résultat qui compte physiquement : on n'a pas introduit un champ auxiliaire gratuit, on a introduit un champ dont la **propre** dynamique est invariante de jauge.

## 5. Structure SU(2) : matrices de Pauli et constantes de structure

Chez Yang–Mills, le groupe n'est plus le cercle U(1) mais **SU(2)** (le spin isotopique du papier de 1954). Les générateurs $T^a = \sigma^a/2$ ($\sigma^a$ matrices de Pauli) obéissent à

$$[T^a, T^b] = i\,\varepsilon^{abc}\, T^c,$$

les $\varepsilon^{abc}$ sont les **constantes de structure** — c'est cette algèbre non abélienne qui va fabriquer l'auto-interaction.

In [6]:
# Verification exhaustive des 9 commutateurs [sigma_a/2, sigma_b/2]
s1 = sp.Matrix([[0, 1], [1, 0]])
s2 = sp.Matrix([[0, -sp.I], [sp.I, 0]])
s3 = sp.Matrix([[1, 0], [0, -1]])
sig = [s1, s2, s3]

def eps(a, b, c):
    # epsilon de Levi-Civita, calcule par parite de permutation
    return sp.LeviCivita(a, b, c)

ok = 0
for a in range(3):
    for b in range(3):
        comm = sig[a] / 2 * sig[b] / 2 - sig[b] / 2 * sig[a] / 2
        attendu = sp.I * sum((eps(a, b, c) * sig[c] / 2 for c in range(3)),
                              sp.zeros(2, 2))
        if sp.simplify(comm - attendu) == sp.zeros(2, 2):
            ok += 1
print(f"[sigma_a/2, sigma_b/2] = i eps_abc sigma_c/2 : {ok}/9 paires verifiees")

[sigma_a/2, sigma_b/2] = i eps_abc sigma_c/2 : 9/9 paires verifiees


## 6. La courbure non abélienne : le commutateur $[D_\mu, D_\nu]$

En SU(2), la dérivée covariante devient **matricielle** : $D_\mu = \partial_\mu + ig W^a_\mu T^a$ où les trois champs $W^a_\mu$ ($a = 1,2,3$) forment un vecteur isotopique. Le tenseur de courbure se définit par le commutateur :

$$[D_\mu, D_\nu] = ig\, F^a_{\mu\nu}\, T^a, \qquad
F^a_{\mu\nu} = \partial_\mu W^a_\nu - \partial_\nu W^a_\mu \;\underline{- g\, \varepsilon^{abc} W^b_\mu W^c_\nu}.$$

**C'est le cœur de Yang–Mills 1954** : dans la copie archivée (p. 193), le tenseur du papier, noté $f_{\mu\nu} = \partial b_\mu/\partial x_\nu - \partial b_\nu/\partial x_\mu + 2g\, b_\mu \times b_\nu$, porte le même terme bilinéaire $b_\mu \times b_\nu = \varepsilon^{abc} b^b b^c$ — le champ de jauge **s'auto-interagit**, même sans matière.

Les deux écritures n'ont pas le même signe, et c'est normal : la copie archivée emploie la convention $D_\mu = \partial_\mu - ig\,b^a\sigma^a$ (couplage de signe opposé, générateurs $\sigma^a$ non divisés par 2), là où ce carnet fixe $D_\mu = \partial_\mu + ig\,W^a_\mu T^a$ avec $T^a = \sigma^a/2$. Le signe du terme bilinéaire suit la convention de couplage, pas la physique. Plutôt que de le supposer, on le **vérifie** : le commutateur calculé plus bas doit reproduire la formule écrite, terme bilinéaire compris.

In [7]:
# Commutateur exact [D_t, D_x] Psi pour un spineur generique a 2 composantes
W = {  # W[a][mu] : composantes du champ de jauge SU(2), fonctions de (t, x)
    (1, 0): sp.Function('W1t', real=True)(t, x), (1, 1): sp.Function('W1x', real=True)(t, x),
    (2, 0): sp.Function('W2t', real=True)(t, x), (2, 1): sp.Function('W2x', real=True)(t, x),
    (3, 0): sp.Function('W3t', real=True)(t, x), (3, 1): sp.Function('W3x', real=True)(t, x),
}

def Wmat(mu):
    # matrice 2x2  W^a_mu T^a = W^a_mu sigma_a / 2
    return sum((W[(a + 1, mu)] * sig[a] / 2 for a in range(3)), sp.zeros(2, 2))

def D(Psi, mu):
    d = sp.diff(Psi, t) if mu == 0 else sp.diff(Psi, x)
    return sp.expand(d + sp.I * g * Wmat(mu) * Psi)

psi1 = sp.Function('psi1', real=True)(t, x)
psi2 = sp.Function('psi2', real=True)(t, x)
Psi = sp.Matrix([psi1, psi2])

# [D_t, D_x] = D_t D_x - D_x D_t : l'ORDRE compte. Ecrire D(D(Psi,0),1) - D(D(Psi,1),0)
# calculerait [D_x, D_t], qui est l'oppose, et le reste serait non nul alors
# que la formule de F est correcte.
commutateur = sp.expand(D(D(Psi, 1), 0) - D(D(Psi, 0), 1))

# Formule attendue : [D_t, D_x] = i g F^a_{tx} sigma_a/2 avec F^a = d_t W^a_x - d_x W^a_t - g eps W W
F_composantes = {}
for a in range(1, 4):
    derivee = sp.diff(W[(a, 1)], t) - sp.diff(W[(a, 0)], x)
    bilineaire = -g * sum(sp.LeviCivita(a, b, c) * W[(b, 0)] * W[(c, 1)]
                          for b in range(1, 4) for c in range(1, 4))
    F_composantes[a] = sp.expand(derivee + bilineaire)

attendu = sp.expand(sp.I * g
                    * sum((F_composantes[a + 1] * sig[a] / 2 for a in range(3)),
                           sp.zeros(2, 2)) * Psi)
reste = sp.simplify(sp.expand(commutateur - attendu))
print("[D_t, D_x] - i g F^a sigma_a/2 applique a Psi (doit etre nul) :")
print("  composante 1 :", sp.simplify(reste[0, 0]))
print("  composante 2 :", sp.simplify(reste[1, 0]))
print("Terme d'auto-interaction extrait de F^1 :",
      sp.expand(-g * sum(sp.LeviCivita(1, b, c) * W[(b, 0)] * W[(c, 1)]
                         for b in range(1, 4) for c in range(1, 4))))

[D_t, D_x] - i g F^a sigma_a/2 applique a Psi (doit etre nul) :


  composante 1 : 0
  composante 2 : 0
Terme d'auto-interaction extrait de F^1 : -g*W2t(t, x)*W3x(t, x) + g*W2x(t, x)*W3t(t, x)


### Lecture du résultat

Le reste est **nul sur les deux composantes** : `composante 1 : 0`, `composante 2 : 0`. Le commutateur $[D_t,D_x]$ calculé à partir de la seule définition $D_\mu = \partial_\mu + igW^a_\mu T^a$ reproduit donc exactement $igF^a_{\mu\nu}T^a$, terme bilinéaire compris. La formule de la section 6 n'est pas une hypothèse que l'on vérifie : elle **tombe** de l'algèbre.

Le terme d'auto-interaction extrait de $F^1$ vaut `-g*W2t(t, x)*W3x(t, x) + g*W2x(t, x)*W3t(t, x)` : ce sont les deux champs $W^2$ et $W^3$ qui composent la courbure du champ $W^1$. Ce terme n'apparaît dans l'algèbre que parce que les générateurs ne commutent pas — c'est lui qui donne au Lagrangien de Yang–Mills $-\tfrac14 F^a_{\mu\nu}F^{a\,\mu\nu}$ son contenu propre, et qui interdit de traiter $SU(2)$ comme un simple « $U(1)$ en plus dimensionnel ».

Sous une transformation de jauge $S(x,t) \in$ SU(2), le papier montre que « $f_{\mu\nu}$ transforms like a vector » (représentation adjointe) — donc $F^a_{\mu\nu} F^a_{\mu\nu}$ (équivalent de $\mathrm{Tr}\,F_{\mu\nu}F_{\mu\nu}$) est invariant, et c'est lui qui fournit le Lagrangien de Yang–Mills $-\tfrac14 F^a_{\mu\nu}F^{a\,\mu\nu}$.

## 7. Contrôle numérique : $\mathrm{Tr}(F^2)$ invariant sous une jauge SU(2) explicite

**Hypothèse pré-enregistrée** : pour une transformation $U(x,t) = e^{i\theta^a(x,t)T^a}$ et des champs lisses $W^a_\mu$, la quantité $s = \mathrm{Tr}(F_{tx} F_{tx}^\dagger)$ est **inchangée** (écart relatif $< 10^{-12}$) quand on remplace $W_\mu \to W'_\mu = U W_\mu U^\dagger + \tfrac{i}{g}(\partial_\mu U)U^\dagger$ et qu'on recalcule $F$ par la même formule.

In [8]:
# Controle numerique : invariance de Tr(F F) sous jauge SU(2) explicite.
# Tout est derivé SYMBOLIQUEMENT puis lambdifié une seule fois : aucune
# difference finie, la precision plafonne donc au rounding float64.
import numpy as np

# Choix lisse et fixe des fonctions theta^a(t,x) et W^a_mu(t,x)
th1e = 0.3 * sp.sin(2 * t + x)
th2e = 0.2 * sp.cos(t - 3 * x)
th3e = 0.25 * sp.sin(x) * sp.cos(t)
W1te, W1xe = 0.4 * sp.cos(t + x), 0.3 * sp.sin(2 * x)
W2te, W2xe = 0.2 * sp.sin(t) * sp.cos(x), 0.5 * sp.cos(3 * t + x)
W3te, W3xe = 0.35 * sp.exp(-(t - 1)**2 - x**2), 0.15 * sp.sin(t * x)

g_val = sp.Rational(7, 10)  # g = 0.7, exact en rationnel
T1, T2, T3 = s1 / 2, s2 / 2, s3 / 2

# U en forme fermee SU(2), identite de Rodrigues. Avec X = theta.T et
# T^a = sigma^a/2, on a X^2 = (r^2/4) I (r = |theta|), donc
# exp(iX) = cos(r/2) I + i sin(r/2) X/(r/2) : le coefficient porte 2/r.
# Ecrire sin(r/2)/r donnerait une matrice NON UNITAIRE -- piege mesure,
# qui se paie par un ecart d'invariance de 7e-2 au lieu de 1e-16.
r = sp.sqrt(th1e**2 + th2e**2 + th3e**2)
thT = th1e * T1 + th2e * T2 + th3e * T3
U = sp.cos(r / 2) * sp.eye(2) + sp.I * 2 * sp.sin(r / 2) / r * thT
# U^dag en forme fermee : les T^a sont hermitiens, donc
# U^dag = cos(r/2) I - i sin(r/2)/r (theta.T). La transposee seule ne suffit
# PAS (sigma_2 est antisymetrique) -- piege classique.
Udag = sp.cos(r / 2) * sp.eye(2) - sp.I * 2 * sp.sin(r / 2) / r * thT

# Controle de la forme fermee : exp(i theta.T) doit etre UNITAIRE. C'est le
# filet qui attrape le piege du coefficient 2/r -- une matrice non unitaire
# brise l'invariance de Tr(F F^dag) meme si la transformation est correcte.
_U_f = sp.lambdify((t, x), U, 'numpy')
_Ud_f = sp.lambdify((t, x), Udag, 'numpy')
_pt = (0.37, 0.81)
_ecart_u = np.max(np.abs(_U_f(*_pt) @ _Ud_f(*_pt) - np.eye(2)))
print(f"Unitarite de U au point {_pt} : ||U Udag - I|| = {_ecart_u:.2e}")

Wt_m = W1te * T1 + W2te * T2 + W3te * T3
Wx_m = W1xe * T1 + W2xe * T2 + W3xe * T3
# Transformation de jauge : W'_mu = U W_mu U^dag + (i/g) (d_mu U) U^dag
Wt_p = sp.expand(U * Wt_m * Udag + sp.I / g_val * sp.diff(U, t) * Udag)
Wx_p = sp.expand(U * Wx_m * Udag + sp.I / g_val * sp.diff(U, x) * Udag)

def composantes(Wmat):
    # W'^a = Tr(2 T^a W) ; les trois T de trace 1/2 extraient les composantes
    return tuple(sp.expand(2 * sp.trace(Ta * Wmat)) for Ta in (T1, T2, T3))

def F_comps(Wt_c, Wx_c):
    # F^a_tx = d_t W^a_x - d_x W^a_t - g eps_abc W^b_t W^x_c (formule du §6)
    return tuple(
        sp.expand(sp.diff(Wx_c[a], t) - sp.diff(Wt_c[a], x)
                  - g_val * sum(sp.LeviCivita(a + 1, b + 1, c + 1) * Wt_c[b] * Wx_c[c]
                                for b in range(3) for c in range(3)))
        for a in range(3))

F0 = F_comps((W1te, W2te, W3te), (W1xe, W2xe, W3xe))
F1 = F_comps(composantes(Wt_p), composantes(Wx_p))

# Tr(F F^dag) = F^a F^b Tr(T^a T^b) = (1/2) sum_a (F^a)^2
s_avant = sp.Rational(1, 2) * sum(Fa**2 for Fa in F0)
s_apres = sp.Rational(1, 2) * sum(Fa**2 for Fa in F1)  # pas de nom s1 : collision Pauli
s_avant_f = sp.lambdify((t, x), s_avant, 'numpy')
s_apres_f = sp.lambdify((t, x), s_apres, 'numpy')

grille = [(tt, xx) for tt in np.linspace(0.1, 1.0, 6) for xx in np.linspace(0.1, 1.0, 6)]
ecarts = [abs(s_apres_f(tt, xx) - s_avant_f(tt, xx)) / abs(s_avant_f(tt, xx)) for tt, xx in grille]
print(f"Invariance (1/2) sum (F^a)^2 = Tr(F F^dag) : ecart relatif max = {max(ecarts):.2e} sur {len(grille)} points")
print("Verdict hypothese pre-enregistree (< 1e-12) :",
      "CONFIRMEE" if max(ecarts) < 1e-12 else "REJETEE")

Unitarite de U au point (0.37, 0.81) : ||U Udag - I|| = 3.12e-18


Invariance (1/2) sum (F^a)^2 = Tr(F F^dag) : ecart relatif max = 3.92e-15 sur 36 points
Verdict hypothese pre-enregistree (< 1e-12) : CONFIRMEE


### Lecture du résultat

Trois mesures, et une seule conclusion possible.

1. **Unitarité** : $\|U U^\dagger - I\| = 3.12e-18$ au point $(0.37, 0.81)$ — la transformation appartient bien à SU(2). Ce contrôle n'est pas décoratif : la forme de Rodrigues doit porter le coefficient $2/r$ (et non $1/r$) pour $T^a = \sigma^a/2$, faute de quoi la matrice n'est pas unitaire et l'écart d'invariance passe de $10^{-15}$ à $7e-2$ — onze ordres de grandeur au-dessus du seuil pré-enregistré. Le seuil a fait son travail : il a **rejeté** une version fausse du calcul.
2. **Invariance** : l'écart relatif maximal entre $\tfrac12\sum_a (F^a)^2$ et $\mathrm{Tr}(F F^\dagger)$ vaut `3.92e-15` sur les 36 points de la grille — soit **environ deux ordres et demi** sous le seuil pré-enregistré de $10^{-12}$. L'hypothèse est **CONFIRMÉE**.
3. **Sens du test** : l'écart est petit mais non nul — c'est l'arrondi IEEE-754, pas une identité symbolique. Une identité exacte donnerait 0. La mesure confirme donc l'invariance **à la précision de la double précision**, et pas au-delà : c'est la limite de l'outillage numérique, pas une réserve sur la physique.

## 8. Contre-témoin : transformer comme si le groupe était abélien

Si l'on applique à SU(2) la **seule** règle abélienne (translation $W_\mu \to W_\mu + \tfrac{1}{g}\partial_\mu\theta$, sans le terme $U W_\mu U^\dagger$), l'invariance doit **se rompre** — c'est le témoin que la section précédente mesure bien la structure non abélienne et pas une identité triviale.

In [9]:
# Contre-temoin : transformation abelienne appliquee au champ non abelien.
# On garde les memes theta^a et la meme grille qu'au §7 ; seule la regle de
# transformation change : simple translation W_mu -> W_mu + (1/g) d_mu theta,
# sans la rotation adjointe U W_mu U^dag. Tout le reste du pipeline est identique.
Wt_ct = tuple(sp.expand(Wte + sp.diff(the, t) / g_val)
              for Wte, the in zip((W1te, W2te, W3te), (th1e, th2e, th3e)))
Wx_ct = tuple(sp.expand(Wxe + sp.diff(the, x) / g_val)
              for Wxe, the in zip((W1xe, W2xe, W3xe), (th1e, th2e, th3e)))
F_ct = F_comps(Wt_ct, Wx_ct)  # formule du §6 appliquee aux champs translates
s_ct = sp.Rational(1, 2) * sum(Fa**2 for Fa in F_ct)
s_ct_f = sp.lambdify((t, x), s_ct, 'numpy')

ecarts_ct = [abs(s_ct_f(tt, xx) - s_avant_f(tt, xx)) / abs(s_avant_f(tt, xx)) for tt, xx in grille]
print(f"Contre-temoin abelien : ecart relatif max = {max(ecarts_ct):.2e} (attendu >> 1e-6)")
print("La transformation incomplete est DETECTEE :", "OUI" if max(ecarts_ct) > 1e-6 else "NON")

Contre-temoin abelien : ecart relatif max = 8.16e-01 (attendu >> 1e-6)
La transformation incomplete est DETECTEE : OUI


## 9. Exercices

Trois exercices pour prolonger, dans l'esprit des précédents : le carnet doit s'exécuter de bout en bout même non complété.

### Exercice 1 — Jauge U(1) sur le champ de Dirac
Écrire (en représentation matricielle sympy) le Lagrangien de Dirac couplé $\bar\psi(i\gamma^\mu D_\mu - m)\psi$ et vérifier l'invariance de jauge locale.

In [10]:
# Exercice 1 -- a completer
# Indice : gamma^0 = [[0, I],[I, 0]], gamma^1 = [[0, -I],[I, 0]] en representation chirale 1+1 D
# Etape 1 : construire psi comme vecteur sympy a 2 composantes complexes
# Etape 2 : former D_mu psi avec le A_mu du carnet
# Etape 3 : verifier que la transformation conjointe laisse L invariant
resultat_ex1 = None  # TODO etudiant
print("Exercice a completer : jauge U(1) du champ de Dirac")

Exercice a completer : jauge U(1) du champ de Dirac


### Exercice 2 — Doublet scalaire SU(2)
Le champ de Higgs avant symmetry-breaking : un doublet $\Phi = (\phi_1, \phi_2)^T$ de scalaires complexes. Construire $D_\mu\Phi$ avec les $W^a_\mu$ du carnet et vérifier l'invariance de $(D_\mu\Phi)^\dagger(D^\mu\Phi)$ sous $S(x,t)\in$ SU(2).

In [11]:
# Exercice 2 -- a completer
# Indice : Phi' = S Phi, W'_mu = S W_mu S^dag + (i/g) (d_mu S) S^dag
# Etape 1 : poser S = exp(i theta^a T^a) avec les theta du carnet
# Etape 2 : developper (D Phi)^dag (D Phi) avant/apres
resultat_ex2 = None  # TODO etudiant
print("Exercice a completer : doublet scalaire SU(2)")

Exercice a completer : doublet scalaire SU(2)


### Exercice 3 — Composante $F^3$ explicite
Pour $W^1_t = W^2_x = 0$ et des champs $W^3_t, W^3_x, W^2_t, W^1_x$ quelconques, calculer $F^3_{tx}$ à la main (formule du §6) puis le vérifier avec le commutateur du carnet. Que reste-t-il quand $g \to 0$ ?

In [12]:
# Exercice 3 -- a completer
# Indice : F^3 = d_t W^3_x - d_x W^3_t - g (W^1_t W^2_x - W^2_t W^1_x)
# Etape 1 : poser les fonctions, calculer la formule
# Etape 2 : comparer au commutateur [D_t, D_x] du §6
resultat_ex3 = None  # TODO etudiant
print("Exercice a completer : composante F^3 et limite abelienne")

Exercice a completer : composante F^3 et limite abelienne


## 10. Périmètre honnête

- Ce carnet vérifie l'invariance de jauge **classique** (niveau Lagrangien), par calcul symbolique exact (§3, §4, §6) et contrôle numérique (§7). Il ne dit rien de la quantification, des fantômes de Faddeev-Popov, ni de la renormalisabilité — le papier de 1954 lui-même s'arrête avant.
- Le **trou de masse** (pourquoi les $W^a$ semblent sans masse dans $-\tfrac14 F^2$ alors que la force nucléaire est de courte portée) est le problème du prix du Millénaire Clay — la description officielle Jaffe–Witten est archivée au gisement, et le résultat numérique d'OpenAI sur Yang–Mills reste en veille : aucun carnet de cette série ne l'énonce. La veille est tracée dans l'EPIC #23161.
- La vérification de §7 valide la **cohérence interne** du pipeline (formule de $F$, transformation, re-calcul) sur des champs lisses choisis ; ce n'est ni une preuve formelle ni un balayage exhaustif de l'espace des champs.
- **Jauge-03** (à venir) posera la même structure sur **réseau de Wilson** : la vérité plantée en dimension 2 et les rapports de Creutz.

## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Phase globale → locale | $\delta L = 0$ pour $\alpha$ **constant** ($§2$) ; la phase locale $\alpha(x,t)$ fait surgir des $\partial\alpha$ dans $\delta L$ — l'invariance n'est pas gratuite | §2, §3 |
| Dérivée covariante | $D'_\mu\phi' = e^{i\alpha}D_\mu\phi$ : le champ de jauge absorbe les $\partial\alpha$, alors $F' - F = 0$ et $\delta L_{\text{total}} = 0$, tout en symbolique exact | §3, §4, §5 |
| Non-abélien | $[\sigma_a/2, \sigma_b/2] = i\varepsilon_{abc}\sigma_c/2$ vérifié sur **9/9 paires** ; $[D_t, D_x] - igF^a\sigma_a/2 = 0$ sur les deux composantes — la formule de $F$ **tombe** de l'algèbre, elle n'est pas posée | §6 |
| Auto-interaction | le terme $-g\,W^2_t W^3_x + g\,W^2_x W^3_t$ vit **dans** $F^1$ : ce sont $W^2$ et $W^3$ qui composent la courbure de $W^1$ — impossible en abélien | §6 |
| Contrôle numérique | unitarité $\|UU^\dagger - I\| = 3{,}12 \times 10^{-18}$ ; $\tfrac12\sum_a (F^a)^2 = \mathrm{Tr}(FF^\dagger)$ à $3{,}92 \times 10^{-15}$ sur 36 points — hypothèse pré-enregistrée ($< 10^{-12}$) : **CONFIRMÉE** | §7 |
| Contre-témoin | appliquer la **seule** règle abélienne à SU(2) (translation sans le terme $UWU^\dagger$) rompt l'invariance : écart $8{,}16 \times 10^{-1}$, transformation incomplète **DÉTECTÉE** | §8 |


## Vérifiez votre compréhension

1. La phase **globale** laisse $\delta L = 0$ sans rien demander. Qu'est-ce que la phase **locale** fait surgir, et que faut-il inventer pour le neutraliser ?

<details><summary>Réponse</summary>

Elle fait surgir les dérivées $\partial_\mu\alpha$ : l'expression complète de $\delta L$ (§3) ne s'annule plus, chaque $\partial\alpha$ (y compris les termes quadratiques en $\partial\alpha$) traîne dans le calcul. Pour neutraliser : promouvoir $\partial_\mu$ en $D_\mu = \partial_\mu + igA_\mu$ et laisser le champ de jauge se transformer pour absorber les $\partial\alpha$. Alors $D'_\mu\phi' = e^{i\alpha}D_\mu\phi$, $F'-F = 0$ et $\delta L_{\text{total}} = 0$ — vérifié en symbolique exact (§4, §5).

</details>

2. D'où vient le terme d'auto-interaction des $W^a$, et pourquoi l'électromagnétisme n'en a-t-il pas ?

<details><summary>Réponse</summary>

Il vit **dans** la définition de la courbure : le terme $-g\,W^2_t W^3_x + g\,W^2_x W^3_t$ est extrait de $F^1$ (§6) — ce sont $W^2$ et $W^3$ qui composent la courbure du champ $W^1$. Sa cause est la non-commutativité : $[\sigma_a/2, \sigma_b/2] = i\varepsilon_{abc}\sigma_c/2$ (9/9 paires vérifiées), et le calcul de $[D_t, D_x]$ reproduit exactement $igF^a\sigma_a/2$, terme bilinéaire compris. En abélien, les générateurs commutent, $F$ reste linéaire en $A$, et aucune composante du champ ne compose la courbure d'une autre.

</details>

3. À quoi sert le contre-témoin du §8, et que prouve son écart de $8{,}16 \times 10^{-1}$ ?

<details><summary>Réponse</summary>

Il applique à SU(2) la **seule** règle abélienne ($W_\mu \to W_\mu + \tfrac1g\partial_\mu\theta$, sans le terme $UW_\mu U^\dagger$) : si l'invariance survivait, c'est que le contrôle du §7 mesurait une identité triviale — le vert de l'hypothèse pré-enregistrée ($< 10^{-12}$) ne testerait alors rien. L'écart explose à $8{,}16 \times 10^{-1}$ (attendu $\gg 10^{-6}$), verdict « transformation incomplète **DÉTECTÉE** » : le contrôle du §7 mesure bien la structure non abélienne.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Jauge-03 — Action de Wilson et confinement](Jauge-03-Action-Wilson-Confinement-Python.ipynb) — la même structure posée sur **réseau** : action de Wilson, boucles en dimension 2 exactement soluble (vérité plantée contre la source archivée), loi de l'aire et rapports de Creutz.
- **Carnet précédent** : [Jauge-01 — Théorème de Noether](Jauge-01-Theoreme-Noether-Python.ipynb) — le socle : symétrie continue ⇒ courant conservé, mesuré deux fois (symbolique et DOP853) avec contre-témoin.
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161) — l'arc Noether → Yang–Mills ; le trou de masse (Clay, Jaffe–Witten archivé au gisement) y est tracé en veille.


## Références — copies archivées au gisement

Toute affirmation attribuée à une source ci-dessous a été écrite **depuis la copie archivée** au gisement `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\` :

- **Yang & Mills 1954** — `1954 - Yang Mills - Conservation of Isotopic Spin and Isotopic Gauge Invariance (PR 96, 1954).pdf` : définition de la jauge isotopique (p. 192), tenseur $f_{\mu\nu}$ avec terme bilinéaire et transformation vectorielle (p. 193), Lagrangien en $f\cdot f$.
- **Jaffe & Witten 2000** — `2000 - Jaffe et Witten - Quantum Yang-Mills Theory (Description officielle Clay Mathematics Institute, 2000).pdf` : le problème du trou de masse (périmètre §10).
- **Noether 1918** (trad. Tavel) — voir [Jauge-01](Jauge-01-Theoreme-Noether-Python.ipynb).
- Schwartz (*QFT and the Standard Model*) : **non utilisé** — aucune copie libre légale disponible ; ce carnet s'appuie sur les sources primaires seules (question Q33 au registre user en attente de réponse).